# Diagnóstico de Qualidade

## 1. Leitura dos dados da camada Bronze


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.bronze.vb_matches")
colunas = [c for c in df.columns if not c.startswith("_")]  # ignora os metadados técnicos

print(f"{df.count():,} linhas | {len(colunas)} colunas de dados")


76,756 linhas | 65 colunas de dados


## 2. Unicidade

Procura a chave natural que identifica cada partida e verifica se há duplicatas.

### 2.1. Chaves candidatas


In [0]:
from itertools import combinations
from pyspark.sql import functions as F

# Unidades candidatas a compor a chave. Atletas entram como bloco: só fazem sentido os 4 juntos.
# year fica fora: é derivado de campo date.
unidades = {
    "circuit":    ["circuit"],
    "tournament": ["tournament"],
    "date":       ["date"],
    "gender":     ["gender"],
    "bracket":    ["bracket"],
    "round":      ["round"],
    "match_num":  ["match_num"],
    "atletas":    ["w_player1", "w_player2", "l_player1", "l_player2"],
}
total = df.count()

# Todas as 255 combinações das unidades, com rótulo e colunas de cada uma
combos = [c for n in range(1, len(unidades) + 1) for c in combinations(unidades, n)]
nomes = {c: " + ".join(c) for c in combos}
cols  = {c: [col for u in c for col in unidades[u]] for c in combos}

# Contagem distintas de todas as combinações, numa única agregação
exprs = [F.countDistinct(*cols[c]).alias(nomes[c]) for c in combos]
contagens = df.agg(*exprs).collect()[0].asDict()

# Chave única: cada combinação de valores aparece em uma linha só (distintas == total)
unicas  = [c for c in combos if contagens[nomes[c]] == total]

# Chave mínima: única e, se tirar qualquer coluna, deixa de ser única
minimas = [c for c in unicas if not any(set(o) < set(c) for o in unicas)]
minimas = sorted(minimas, key=lambda c: len(cols[c]))

resultado = spark.createDataFrame(
    [(nomes[c], len(cols[c]), contagens[nomes[c]], c in unicas, c in minimas) for c in combos],
    ["chave", "n_colunas", "distintas", "unica", "minima"],
)

print(f"{len(unicas)} únicas de {len(combos)} combinações")
print("mínimas:", *[f"{nomes[c]}  ({len(cols[c])} colunas)" for c in minimas], sep="\n  ")
display(resultado.filter("unica and n_colunas <= 7").orderBy("n_colunas", "chave"))  # coloquei filter para deixar mais enxuto

36 únicas de 255 combinações
mínimas:
  tournament + date + gender + bracket + match_num  (5 colunas)
  date + match_num + atletas  (6 colunas)


chave,n_colunas,distintas,unica,minima
tournament + date + gender + bracket + match_num,5,76756,true,true
circuit + tournament + date + gender + bracket + match_num,6,76756,true,false
date + match_num + atletas,6,76756,true,true
tournament + date + gender + bracket + round + match_num,6,76756,true,false
circuit + date + match_num + atletas,7,76756,true,false
circuit + tournament + date + gender + bracket + round + match_num,7,76756,true,false
date + bracket + match_num + atletas,7,76756,true,false
date + gender + match_num + atletas,7,76756,true,false
date + round + match_num + atletas,7,76756,true,false
tournament + date + match_num + atletas,7,76756,true,false


**Resultado.** Das 255 combinações, 36 são únicas e apenas duas são mínimas:

- `tournament + date + gender + bracket + match_num` —> 5 colunas, pela estrutura do torneio
- `date + match_num + 4 atletas` —> 6 colunas, pelos participantes

As outras 34 únicas são superconjuntos de uma dessas duas.

**Chave adotada:** `tournament + date + gender + bracket + match_num`. Além de menor,
ela não depende de quem jogou, e os dados mostram por que isso importa: a chave pelos
participantes só fecha com `match_num`, porque as mesmas quatro atletas podem se
enfrentar duas vezes na mesma rodada (Tenerife 2001: um par de partidas com placar e
duração diferentes, um playoff). Atletas são atributo da partida, não identidade. Além
disso, nomes chegam com inconsistências (espaço duplo, apelidos entre aspas), o que
tornaria a chave frágil.

In [0]:
# Única colisão da chave pelos participantes sem match_num: mesmas atletas, mesma rodada
chave_atletas = ["date", "bracket", "round", "w_player1", "w_player2", "l_player1", "l_player2"]

par = (
    df.groupBy(*chave_atletas).count().filter("count > 1")
    .join(df, chave_atletas)
    .select("tournament", "date", "bracket", "round", "match_num", "score", "duration",
            "w_player1", "w_player2", "l_player1", "l_player2")
    .orderBy("match_num")
)
display(par)

tournament,date,bracket,round,match_num,score,duration,w_player1,w_player2,l_player1,l_player2
Tenerife,2001-06-13,Qualifier Playoff,NA,74,"21-23, 21-16, 15-7",00:53:00,Adriano Garrido,Marcio Araujo,Jefferson Bellaguarda,Paulao Moreira
Tenerife,2001-06-13,Qualifier Playoff,NA,75,"18-21, 21-18, 15-12",00:53:00,Adriano Garrido,Marcio Araujo,Jefferson Bellaguarda,Paulao Moreira


### 2.2. Duplicatas exatas


In [0]:
# Linhas totalmetne repetidas em todas as colunas
duplicatas = df.select(*colunas).count() - df.select(*colunas).distinct().count()
print(f"duplicatas exatas: {duplicatas}")

duplicatas exatas: 0


Zero duplicatas. A fonte não repete linhas; nenhuma remoção é necessária durante o tratamento.

## 3. Valores ausentes

In [0]:
total = df.count()

# Para cada coluna, duas contagens: quantas linhas são NULL e quantas têm o texto "NA".
# Tudo numa única agregação, em vez de um count() por coluna.
exprs = []
for c in colunas: 
    exprs.append(F.sum(F.col(c).isNull().cast("int")).alias(f"{c}__nulos"))
    exprs.append(F.sum((F.col(c) == "NA").cast("int")).alias(f"{c}__na"))

# Devolve uma única linha com todas as contagens
contagens = df.agg(*exprs).collect()[0].asDict()

# Reorganiza em uma linha por coluna: nome, NULLs, "NA"s
completude = spark.createDataFrame(
    [(c, contagens[f"{c}__nulos"], contagens[f"{c}__na"]) for c in colunas],
    ["coluna", "nulos", "na_texto"]
)

# Adiciona o percentual ausente => (NULL + "NA")/total de linhas
completude = completude.withColumn(
    "pct_ausente", F.round((F.col("nulos") + F.col("na_texto")) / total * 100, 1)
)

display(completude.orderBy(F.desc("pct_ausente")))

coluna,nulos,na_texto,pct_ausente
w_p1_tot_errors,0,62413,81.3
w_p1_tot_serve_errors,0,62417,81.3
w_p2_tot_errors,0,62413,81.3
w_p2_tot_serve_errors,0,62413,81.3
l_p1_tot_errors,0,62413,81.3
l_p1_tot_serve_errors,0,62418,81.3
l_p2_tot_errors,0,62413,81.3
l_p2_tot_serve_errors,0,62417,81.3
w_p1_tot_attacks,0,62178,81.0
w_p1_tot_kills,0,62178,81.0


**Resultado.** Nenhuma coluna tem NULL real: toda ausência é o texto `"NA"`.
A silver converte esse texto em NULL antes de qualquer cast.

Dois padrões de ausência, com tratamentos diferentes:

- **Estrutural (~81% ausente):** as estatísticas de jogo (`*_tot_attacks`, `*_tot_kills`,
  `*_tot_aces`, `*_tot_blocks`, `*_tot_digs`, `*_tot_errors`, `*_hitpct`), com a mesma
  proporção nas quatro posições. Não é falha: a maioria dos torneios não coletou essas
  estatísticas. Com isso, acrescentamos a flag `tem_estatistica` na camada silver, e a pergunta 7 usa só esse subconjunto.
- **Pontual:** altura (9,1% em perdedores, 5,2% em vencedores), `round` (6,4%),
  `duration` (2,9%), `*_birthdate` (~1%) e `score` (22 linhas). Mantemos como NULL;
  as perguntas que dependem dessas colunas filtram a ausência na própria consulta.

## 4. Formatos e valores válidos
### 4.1. Formato dos campos

Uma coluna representante de cada tipo de dado, com o formato esperado. Mede, entre os
valores preenchidos, quantos não seguem o formato, e mostra um exemplo.

In [0]:
# Para uma coluna e um formato esperado (regex): entre os valores preenchidos,
# qual a porcentagem que não segue o formato, e um exemplo desses valores
def fora_padrao(col, regex):
    validas = df.filter(F.col(col) != "NA")                # ignora ausências
    fora = validas.filter(~F.col(col).rlike(regex))        # o que não bate com o formato
    pct = round(fora.count() / validas.count() * 100, 2)
    # dois exemplos: o valor mais curto e o mais longo separado por barra, para mostrar formatos diferentes
    valores = [r[0] for r in fora.select(col).distinct().orderBy(F.length(col)).collect()]
    exemplos = " | ".join([valores[0], valores[-1]]) if valores else None
    return pct, exemplos

  # Regex que o campo deveria ter
formatos = [
      ("date",             r"^\d{4}-\d{2}-\d{2}$"),          # data ISO: 2019-08-29
      ("w_p1_birthdate",   r"^\d{4}-\d{2}-\d{2}$"),          # data ISO
      ("score",            r"^(\d+-\d+)(, \d+-\d+)*$"),      # sets separados por vírgula: 21-19, 18-21, 15-13
      ("duration",         r"^\d{2}:\d{2}:\d{2}$"),          # hh:mm:ss
      ("w_p1_hgt",         r"^\d+$"),                        # inteiro (polegadas)
      ("w_rank",           r"^\d+$"),                        # inteiro (posição no ranking)
      ("match_num",        r"^\d+$"),                        # inteiro; compõe a chave da partida
      ("w_rank",           r"^(\d+|Q\d+|\d+, Q\d+)$"),       # domínio completo do ranking: 7 | Q9 | 24, Q30
      ("l_rank",           r"^(\d+|Q\d+|\d+, Q\d+)$"),       # idem para a dupla perdedora
      ("w_p1_tot_attacks", r"^-?\d+$"),                      # estatística inteira, pode ser negativa
      ("w_p1_tot_hitpct",  r"^-?\d+(\.\d+)?$"),              # estatística decimal com sinal
  ]

display(spark.createDataFrame(
    [(c, r, *fora_padrao(c, r)) for c, r in formatos],
    ["coluna", "padrao_esperado", "pct_fora_do_padrao", "exemplos_fora"]
))

coluna,padrao_esperado,pct_fora_do_padrao,exemplos_fora
date,^\d{4}-\d{2}-\d{2}$,0.0,null
w_p1_birthdate,^\d{4}-\d{2}-\d{2}$,0.0,null
score,"^(\d+-\d+)(, \d+-\d+)*$",1.39,"9-5 retired | 19-21, 21-17, 4-1 retired"
duration,^\d{2}:\d{2}:\d{2}$,0.0,null
w_p1_hgt,^\d+$,0.0,null
w_rank,^\d+$,39.24,"Q9 | 24, Q30"
match_num,^\d+$,0.0,null
w_rank,"^(\d+|Q\d+|\d+, Q\d+)$",0.0,null
l_rank,"^(\d+|Q\d+|\d+, Q\d+)$",0.0,null
w_p1_tot_attacks,^-?\d+$,0.0,null


**Resultado.** Data, nascimento, duração, altura, `match_num` e as estatísticas de jogo
seguem um único formato (0% fora). Dois campos fogem do padrão de propósito:

- `score` (1,39% fora): `Forfeit or other` e placar parcial seguido de `retired`
  (`9-5 retired`, `19-21, 21-17, 4-1 retired`), partidas não concluídas. Viram a flag
  `partida_incompleta` na camada silver.
- `w_rank` (39,24% fora de inteiro): `Q<n>` é o seed da qualificatória e `24, Q30` traz
  seed principal e da qualificatória no mesmo campo. No tratamento separaremos em `seed_principal`
  e `seed_qualificatoria`, nos dois lados.

Testados contra o conjunto completo (`\d+`, `Q\d+` ou `\d+, Q\d+`) os dois campos de
ranking dão **0% fora**: as três formas cobrem todos os valores. Com isso, toda coluna a ser
convertida tem valor convertível, com exceção da `score`, que ficará como texto e terá
apenas os sets contados, com `try_cast`. Para as demais, a decisão é usar `cast` estrito no
tratamento: um formato novo numa recarga futura deve falhar em vez de virar NULL em silêncio.

### 4.2. Variantes de grafia nas categóricas

Lista os valores distintos de cada coluna categórica. Uma variante de grafia
apareceria como dois valores parecidos (caixa, espaço, apóstrofo) para a mesma fase.

In [0]:
# Valores distintos das categóricas: confere se o conjunto é fechado e sem variantes de grafia
for c in ["circuit", "gender", "bracket"]:
    print(c, "→", [r[0] for r in df.select(c).distinct().orderBy(c).collect()])

circuit → ['AVP', 'FIVB']
gender → ['M', 'W']
bracket → ['3rd Place', 'Bronze Medal', "Contender's Bracket", 'Country Quota Matches', 'Finals', 'Gold Medal', 'Lucky Losers', 'Pool A', 'Pool B', 'Pool C', 'Pool D', 'Pool E', 'Pool F', 'Pool G', 'Pool H', 'Pool I', 'Pool J', 'Pool K', 'Pool L', 'Pool M', 'Pool N', 'Pool O', 'Pool P', 'Pool Q', 'Pool R', 'Pool S', 'Pool T', 'Pool U', 'Pool V', 'Pool W', 'Pool X', 'Pool Y', 'Qualifier Bracket', 'Qualifier Playoff', 'Semifinals', "Winner's Bracket"]


**Resultado.** As três colunas têm domínio fechado e sem variantes de grafia:
`circuit` = {AVP, FIVB}, `gender` = {M, W}, `bracket` = 36 fases de torneio. Nenhuma
precisa de normalização na camada silver. Os 36 valores de `bracket` são agrupados em
`fase` na etapa de tratamento.

## 5. Valores plausíveis

Confere se os valores fazem sentido para o vôlei de praia: nascimento, altura, idade,
duração da partida e placar. 

### 5.1. Faixas observadas

Mínimo e máximo de nascimento, altura e duração, usando a posição `w_p1` como amostra
(as quatro posições têm a mesma origem). `try_cast` converte o texto e devolve NULL
para `"NA"`, então os extremos ignoram ausências.

In [0]:
# Conversões de texto para tipo, sem falhar em "NA"
nasc = F.expr("try_to_date(w_p1_birthdate)")
hgt  = F.expr("try_cast(w_p1_hgt as int)")
dur = F.expr("try_cast(get(split(duration, ':'), 0) as int) * 60 + try_cast(get(split(duration, ':'), 1) as int)")

display(df.select(
    F.min(nasc).alias("nasc_min"), F.max(nasc).alias("nasc_max"),
    F.min(hgt).alias("alt_min_pol"), F.max(hgt).alias("alt_max_pol"),
    F.min(dur).alias("dur_min_min"), F.max(dur).alias("dur_max_min"),
))

# Erro clássico de data: nascimento com século errado
print("nascimentos antes de 1950:", df.filter(nasc < "1950-01-01").count())

nasc_min,nasc_max,alt_min_pol,alt_max_pol,dur_min_min,dur_max_min
1953-06-13,2004-07-15,63,85,2,134


nascimentos antes de 1950: 0


**Resultado.** Nascimentos entre 1953 e 2004: nenhum antes de 1950, ou seja, não há
o erro de século comum em datas. Altura entre 63 e 85 polegadas (1,60 m a 2,16 m),
faixa plausível para atletas; o tratamento convertera para cm. Duração entre 2 e 134 minutos:
o mínimo é impossível para uma partida completa e o máximo é raro mas possível. Partidas
com menos de 15 minutos recebem a flag `duracao_suspeita` na camada silver, sem exclusão.

### 5.2. Idade informada e idade calculada

A fonte traz `*_age` pronta. Recalculamos a idade a partir do nascimento e da data
da partida e contamos onde as duas divergem.

In [0]:
data = F.expr("try_to_date(date)")
age  = F.expr("try_cast(w_p1_age as double)")

# Idade inteira na data da partida, calculada do nascimento, e a informada pela fonte
idade_calc = F.floor(F.datediff(data, nasc) / 365.25)
idade_inf  = F.floor(age)

divergentes = (
    df.withColumn("idade_calc", idade_calc)
      .withColumn("idade_inf", idade_inf)
      .withColumn("diferenca", F.col("idade_inf") - F.col("idade_calc"))
      .filter(F.col("diferenca") != 0)
)

print("idade informada difere da calculada:", divergentes.count())
display(
    divergentes.groupBy("tournament", "date", "diferenca").count().orderBy("date")
)

idade informada difere da calculada: 460


tournament,date,diferenca,count
Puerto Vallarta,2015-10-06,1,85
Puerto Vallarta,2015-10-07,1,82
Antalya,2015-10-20,1,89
Antalya,2015-10-21,1,101
Doha,2015-11-08,1,103


**Resultado.** 460 linhas com idade um ano acima da real, todas em três torneios de
outubro e novembro de 2015. Isso indica que o nascimento está certo e a idade foi possivelmente calculada errado na fonte. Por isso no tratamento descartamos `*_age` e calculamos `idade_na_partida` a partir de
`nascimento` e `data`.

### 5.3. Duração da partida

In [0]:
# Distribuição da duração em minutos (dur definida em 5.1; NULL para "NA")
df_dur = df.withColumn("dur_minutos", dur).filter(F.col("dur_minutos").isNotNull())

display(df_dur.select(
    F.count("*").alias("n"),
    F.min("dur_minutos").alias("min"),
    F.expr("percentile(dur_minutos, 0.01)").alias("p01"),
    F.expr("percentile(dur_minutos, 0.50)").alias("mediana"),
    F.expr("percentile(dur_minutos, 0.99)").alias("p99"),
    F.max("dur_minutos").alias("max"),
))

# Extremos: abaixo de 15 min não dá para completar dois sets; acima de 120 é raro
print("partidas com menos de 15 min:", df_dur.filter("dur_minutos < 15").count())
print("das < 15 min, quantas são Forfeit/retired:", df_dur.filter("dur_minutos < 15 and score not rlike '^[0-9]'").count())
print("partidas com mais de 120 min:", df_dur.filter("dur_minutos > 120").count())

n,min,p01,mediana,p99,max
74507,2,26.0,42.0,74.0,134


partidas com menos de 15 min: 39
das < 15 min, quantas são Forfeit/retired: 0
partidas com mais de 120 min: 3


**Resultado.** 74.507 partidas com duração informada (2,9% ausentes). Mediana de 42 min,
com 98% entre 26 e 74 min. Nos extremos, 39 partidas abaixo de 15 min (mínimo de 2 min)
com placar completo, ou seja, duração possivelmente registrada errado, e 3 acima de 120 min (máximo
de 134), raro mas possível. A camada silver marca as abaixo de 15 min com `duracao_suspeita`,
sem excluir, para que as análises de duração possam filtrá-las.

### 5.4. Placar e vencedor

In [0]:
# Sets do placar: "21-19, 18-21, 15-13" → pares (21,19), (18,21), (15,13)
sets = F.expr("regexp_extract_all(score, '(\\\\d+)-(\\\\d+)', 0)")
sets_w = F.expr("size(filter(regexp_extract_all(score, '(\\\\d+)-(\\\\d+)', 0), s -> int(split(s, '-')[0]) > int(split(s, '-')[1])))")
sets_l = F.expr("size(filter(regexp_extract_all(score, '(\\\\d+)-(\\\\d+)', 0), s -> int(split(s, '-')[0]) < int(split(s, '-')[1])))")

placar_completo = r"^(\d+-\d+)(, \d+-\d+)*$"

df_placar = (
    df.filter(F.col("score").rlike(placar_completo))   # exclui Forfeit, retired e NA
      .withColumn("sets_vencedor", sets_w)
      .withColumn("sets_perdedor", sets_l)
)
inconsistentes = df_placar.filter("sets_vencedor <= sets_perdedor or sets_vencedor = 3")

print("placares completos:", df_placar.count())
print("partidas incompletas (Forfeit/retired):", df.filter((F.col("score") != "NA") & ~F.col("score").rlike(placar_completo)).count())
print("placar inconsistente com o vencedor:", inconsistentes.count())
display(inconsistentes.select("tournament", "date", "score", "sets_vencedor", "sets_perdedor").orderBy("date"))

placares completos: 75667
partidas incompletas (Forfeit/retired): 1067
placar inconsistente com o vencedor: 12


tournament,date,score,sets_vencedor,sets_perdedor
Hermosa Beach,2002-06-07,"15-21, 9-7",1,1
Chicago,2005-09-01,"21-14, 24-22, 15-9",3,0
Hermosa Beach,2007-05-18,"22-20, 21-18, 15-8",3,0
Chicago,2015-08-27,"21-19, 22-20, 15-10",3,0
Manhattan Beach,2016-07-14,"21-16, 22-20, 15-13",3,0
Chicago,2016-09-01,"21-14, 22-20, 15-3",3,0
Huntington Beach,2017-05-04,"20-22, 19-21, 15-13",1,2
Austin,2018-05-17,"23-21, 21-18, 16-14",3,0
Manhattan Beach,2018-08-16,"23-21, 21-18, 15-12",3,0
Chicago,2018-08-30,"21-23, 18-21, 15-11",1,2


**Resultado.** 1.067 partidas não têm placar completo (`Forfeit or other` ou `retired`) e
recebem a flag `partida_incompleta` na silver. Entre as 75.667 com placar completo, 12
contradizem o vencedor registrado: 7 em que o vencedor leva os 3 sets (um terceiro set não
deveria existir), 4 em que o perdedor vence 2 sets e 1 partida de set único em que o perdedor
está à frente (`15-21, 9-7`). São erros de registro da fonte, e não há como saber se o errado
é o placar ou o vencedor; essas partidas recebem `placar_inconsistente` e ficam fora das
análises que envolvem placar.

### 5.5. Dupla repetida nos dois lados

In [0]:
# Atletas presentes nos dois lados da mesma linha
repetidos = F.array_intersect(
    F.array("w_player1", "w_player2"),
    F.array("l_player1", "l_player2"),
)

print("partidas com atleta nos dois lados:", df.filter(F.size(repetidos) > 0).count())
display(
    df.filter(F.size(repetidos) > 0)
      .select("circuit", "tournament", "year", "date", "bracket", "score",
              "w_player1", "w_player2", "l_player1", "l_player2")
      .orderBy("date")
)

partidas com atleta nos dois lados: 7


circuit,tournament,year,date,bracket,score,w_player1,w_player2,l_player1,l_player2
FIVB,Marseille,2001,2001-07-16,Qualifier Bracket,Forfeit or other,Heidi Hernandez,Raquel Gonzalez,Heidi Hernandez,Raquel Gonzalez
AVP,Huntington Beach,2010,2010-06-03,Qualifier Bracket,NA,Brian Chapman,John Braunstein,Brian Chapman,John Braunstein
AVP,Huntington Beach,2010,2010-06-03,Qualifier Bracket,Forfeit or other,Jonathan Champeau,Reese Peter,Jonathan Champeau,Reese Peter
AVP,Huntington Beach,2010,2010-06-03,Qualifier Bracket,NA,Kevin Beukema,Kevin Gregan,Kevin Beukema,Kevin Gregan
AVP,St. Petersburg,2013,2013-09-13,Finals,NA,Emily Day,Summer Ross,Emily Day,Summer Ross
FIVB,Aydin,2018,2018-05-17,Pool A,"21-10, 21-9",Lazar Kolaric,Stefan Basta,Lazar Kolaric,Stefan Basta
FIVB,Rubavu,2019,2019-08-22,Pool A,Forfeit or other,Charlotte Nzayisenga,Judith Hakizimana,Charlotte Nzayisenga,Judith Hakizimana


**Resultado.** 7 partidas, todas com a dupla vencedora idêntica à perdedora. Seis não têm
placar (`Forfeit or other` ou ausente): é bye ou W.O., a dupla avançou sem adversário e a
fonte preencheu os dois lados com o mesmo par. A sétima (Aydin 2018, `21-10, 21-9`) tem jogo
real e é erro de registro. Sem correção possível; as seis já recebem `partida_incompleta`, e
a sétima fica documentada. Consequência para o modelo: `(id_partida, id_atleta)` não é chave
única nas participações — o lado (`vencedor`) entra na chave.

## 6. Inventário de problemas e tratamento na silver

| Seção | Problema | Extensão | Tratamento |
|---|---|---|---|
| 2.1 | Fonte sem coluna de id | todas as linhas | `id_partida` = hash de `tournament, date, gender, bracket, match_num` |
| 3 | Ausência codificada como texto `"NA"`, sem NULL real | todas as colunas | converter `"NA"` em NULL antes de qualquer cast |
| 3 | Estatísticas de jogo ausentes | ~81% das partidas, uniforme nas 4 posições (torneio não registrou) | manter NULL; flag `tem_estatistica`; análise restrita a esse subconjunto |
| 3 | Altura ausente | 9,1% perdedores, 5,2% vencedores | manter NULL; excluir da comparação de altura |
| 4.1 | Placar `Forfeit or other` ou `retired` | 1.067 partidas (1,39%) | flag `partida_incompleta`; sets só quando placar completo |
| 4.1 | Ranking nos formatos `Q9` e `24, Q30` | 39,24% dos ranks | separar em `seed_principal` e `seed_qualificatoria` |
| 4.2 | `bracket` com 36 valores | 100% | derivar `fase` (qualificatória, grupos, eliminatória) |
| 5.1 | Altura em polegadas | 100% | converter para cm |
| 5.2 | Idade informada errada (um ano acima) | 460 linhas, três torneios de 2015 | descartar `*_age`; recalcular de `nascimento` e `data` |
| 5.3 | Duração implausível (< 15 min) com placar completo | 39 partidas | flag `duracao_suspeita`, sem excluir |
| 5.4 | Placar contradiz o vencedor | 12 partidas, todas AVP | flag `placar_inconsistente`, sem excluir |
| 5.5 | Dupla vencedora igual à perdedora | 7 partidas (6 bye/W.O. sem placar, 1 erro de registro) | entram em `partida_incompleta`; `vencedor` passa a compor a chave das participações |